In [ ]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import numpy as np


In [ ]:

def split_polygons_by_country(gdf, countries_gdf):
    # Ensure CRS match
    gdf = gdf.to_crs(countries_gdf.crs)
    
    # Create a dictionary to hold subsets
    country_polygons = {country: gpd.GeoDataFrame(columns=gdf.columns, crs=gdf.crs) for country in countries_gdf['name']}
    
    for _, polygon in gdf.iterrows():
        for _, country in countries_gdf.iterrows():
            if polygon.geometry.intersects(country.geometry):
                country_polygons[country['name']] = country_polygons[country['name']].append(polygon, ignore_index=True)
                break

    return country_polygons

def rasterize_polygons(gdf, out_path, width=1000, height=1000):
    # Get the bounds of the GeoDataFrame
    minx, miny, maxx, maxy = gdf.total_bounds

    # Create a transform for the output raster
    transform = from_bounds(minx, miny, maxx, maxy, width, height)

    # Rasterize the GeoDataFrame
    raster = rasterize(
        [(geom, 1) for geom in gdf.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype='uint8'
    )

    # Save the raster to file
    with rasterio.open(
        out_path,
        'w',
        driver='GTiff',
        height=raster.shape[0],
        width=raster.shape[1],
        count=1,
        dtype=raster.dtype,
        crs=gdf.crs,
        transform=transform,
    ) as dst:
        dst.write(raster, 1)


In [ ]:

# Example usage
# polygons_gdf: GeoDataFrame of polygons to split
# countries_gdf: GeoDataFrame of countries with a 'name' column
country_subsets = split_polygons_by_country(polygons_gdf, countries_gdf)

# Rasterize each subset
for country, subset_gdf in country_subsets.items():
    if not subset_gdf.empty:
        rasterize_polygons(subset_gdf, f'{country}_raster.tif')